# CodaBench Submission Package Generator
## Computer Vision Project — UC3M

This notebook packages the final predictions from both model tracks into the CodaBench-compatible `.zip` submission format.

### Submission Format Requirements
The CodaBench competition expects a `.zip` file containing exactly two CSV files:
- **`output_ft.csv`** — predictions from the fine-tuned transfer learning ensemble (1000 rows × 1 column, values in [0,1])
- **`output_custom.csv`** — predictions from the custom architecture ensemble (1000 rows × 1 column, values in [0,1])

Each value is a probability score: 0 = No DR, 1 = DR. CodaBench evaluates using **ROC-AUC**.

### Workflow
1. Load the prediction CSV files generated by the ensemble notebooks
2. Validate format: shape (1000, 1) and values in [0, 1]
3. Export as `output_ft.csv` and `output_custom.csv`
4. Bundle into `codabench_submission.zip`

---

## 1. Load Ensemble Predictions

Final predictions from both ensemble notebooks are loaded:
- **FT ensemble predictions** (`ensemble_submission_mean.csv`): output of the Logistic Regression meta-learner trained on 6 fine-tuned base models
- **Custom model ensemble predictions** (`custom_vgg_multi.csv`): output of the Logistic Regression meta-learner trained on 3 custom models (SmallVGG, LeNet, SmallResNet)

In [6]:
import numpy as np

effnet_path = "predictions/ensemble_submission_mean.csv"
custom_path = "predictions/custom_vgg_multi.csv"
effnet_preds = np.loadtxt(effnet_path, skiprows=1)
custom_preds = np.loadtxt(custom_path)

print("EffNet shape:", effnet_preds.shape)
print("Custom shape:", custom_preds.shape)

EffNet shape: (1000,)
Custom shape: (1000,)


## 2. Prediction Validation

Before packaging, both prediction arrays are validated to ensure they meet CodaBench requirements:
- **Shape:** must be (1000, 1) — one probability per test image
- **Value range:** all values must be in [0, 1] — valid probabilities

Any violation here would cause a CodaBench submission error. Validation is done programmatically to catch issues before upload.

In [7]:
def check_predictions(preds, name):
    print(f"\nChecking {name}...")

    # Shape correcto
    if preds.ndim == 1:
        preds = preds.reshape(-1, 1)

    assert preds.shape[0] == 1000, f"{name} debe tener 1000 filas"
    assert preds.shape[1] == 1, f"{name}debe ser Nx1"

    # Rango correcto
    assert np.all(preds >= 0) and np.all(preds <= 1), \
        f"{name} valores fuera de [0,1]"

    print(f"{name} OK")

    return preds

effnet_preds = check_predictions(effnet_preds, "EffNet")
custom_preds = check_predictions(custom_preds, "Custom")


Checking EffNet...
EffNet OK

Checking Custom...
Custom OK


## 3. Export Predictions to CSV

Predictions are saved as `output_ft.csv` and `output_custom.csv` with 6 decimal places of precision. The file names must exactly match the CodaBench submission format.

In [8]:
np.savetxt("output_ft.csv", effnet_preds, fmt="%.6f")
np.savetxt("output_custom.csv", custom_preds, fmt="%.6f")

## 4. Bundle into Submission ZIP

The two CSV files are packaged into `codabench_submission.zip`. This is the file uploaded directly to the CodaBench platform. The ZIP must contain both files at the root level (no subdirectories).

In [9]:
from zipfile import ZipFile

zip_path = "codabench_submission.zip"

with ZipFile(zip_path, 'w') as zipf:
    zipf.write("output_custom.csv")
    zipf.write("output_ft.csv")

print(f"\nZIP creado en: {zip_path}")


ZIP creado en: codabench_submission.zip


## 5. Submission Verification

Final sanity check: confirm the ZIP contains both required files and that their sizes are consistent with 1000 predictions at 6 decimal places (~9000 bytes each).

In [10]:
import os

print("\nContenido del ZIP:")
with ZipFile(zip_path, 'r') as zipf:
    print(zipf.namelist())

print("\nTamaño archivos:")
print("Custom:", os.path.getsize("output_custom.csv"))
print("FT:", os.path.getsize("output_ft.csv"))


Contenido del ZIP:
['output_custom.csv', 'output_ft.csv']

Tamaño archivos:
Custom: 9000
FT: 9000
